In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder, OrdinalEncoder
from xgboost import XGBRegressor, XGBClassifier
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score


# Data Loading

In [45]:
df = pd.read_csv('D:/Projects/End-to-End/Model/ai_jobs_market_2025_2026.csv')
pd.set_option('display.max_columns', None)
df.head()

,job_id,job_title,job_category,experience_level,years_of_experience,education_required,annual_salary_usd,salary_min_usd,salary_max_usd,city,country,remote_work,company_size,industry,required_skills,ai_salary_premium_pct,demand_score,demand_growth_yoy_pct,benefits_score_10,posting_year,posting_month,is_senior,is_remote_friendly,is_llm_role,salary_tier
0,AIJOB0001,AI Agent Developer,AI Engineering,Senior (6-9 yrs),7,Master's,239000.0,155000,290000,Boston,USA,On-site,Startup (1-50),Finance,APIs|Planning Systems|Python|Cloud|SQL|Leadership,13.1,96,16.9,6.8,2026,3,1,0,1,Senior ($200-300k)
1,AIJOB0002,Prompt Engineer,AI Engineering,Senior (6-9 yrs),2,Bachelor's,166000.0,90000,200000,London,UK,Hybrid,Enterprise (5000+),Finance,Python|Documentation|LLM APIs|Prompt Design|NL...,5.4,82,11.6,6.2,2026,1,1,1,1,Upper-Mid ($150-200k)
2,AIJOB0003,LLM Engineer,AI Engineering,Senior (6-9 yrs),4,Associate's,360000.0,160000,300000,Seattle,USA,Fully Remote,Big Tech (FAANG+),Finance,Vector DBs|Python|Prompt Engineering|Fine-tuni...,9.1,98,42.7,7.7,2026,1,1,1,1,Elite (>$300k)
3,AIJOB0004,Data Engineer (AI),Data Engineering,Senior (6-9 yrs),3,Bachelor's,161000.0,130000,220000,Singapore,Singapore,Fully Remote,SME (51-500),Technology,Feature Stores|Spark|ETL|Airflow|dbt|SQL|Pytho...,12.0,88,6.7,9.5,2026,3,1,1,0,Upper-Mid ($150-200k)
4,AIJOB0005,AI Product Manager,Product,Lead (10+ yrs),5,Bootcamp/Self-taught,283000.0,140000,260000,Los Angeles,USA,Fully Remote,Enterprise (5000+),Automotive,Data Analysis|Stakeholder Mgmt|Agile|Cloud|Pro...,9.4,85,17.3,8.9,2026,1,1,1,0,Senior ($200-300k)


In [46]:
df.nunique()

job_id                   1500
job_title                  25
job_category               12
experience_level            4
years_of_experience        15
education_required          5
annual_salary_usd         248
salary_min_usd             17
salary_max_usd             16
city                       20
country                    14
remote_work                 3
company_size                5
industry                   12
required_skills          1500
ai_salary_premium_pct     151
demand_score               20
demand_growth_yoy_pct     565
benefits_score_10          39
posting_year                2
posting_month              12
is_senior                   2
is_remote_friendly          2
is_llm_role                 2
salary_tier                 5
dtype: int64

In [47]:
df.isnull().sum()

job_id                   0
job_title                0
job_category             0
experience_level         0
years_of_experience      0
education_required       0
annual_salary_usd        0
salary_min_usd           0
salary_max_usd           0
city                     0
country                  0
remote_work              0
company_size             0
industry                 0
required_skills          0
ai_salary_premium_pct    0
demand_score             0
demand_growth_yoy_pct    0
benefits_score_10        0
posting_year             0
posting_month            0
is_senior                0
is_remote_friendly       0
is_llm_role              0
salary_tier              0
dtype: int64

In [48]:
df.describe()

,years_of_experience,annual_salary_usd,salary_min_usd,salary_max_usd,ai_salary_premium_pct,demand_score,demand_growth_yoy_pct,benefits_score_10,posting_year,posting_month,is_senior,is_remote_friendly,is_llm_role
count,1500.000000,1500.000000,1500.000000,1500.000000,1500.000000,1500.000000,1500.000000,1500.000000,1500.000000,1500.000000,1500.000000,1500.000000,1500.000000
mean,6.216000,194892.000000,135448.666667,257537.333333,10.858200,87.523333,31.116333,7.897333,2025.584000,3.968000,0.496667,0.754000,0.218000
std,2.675216,66506.822013,24448.950878,39852.822207,4.029742,8.026315,22.046343,1.102846,0.493058,3.270388,0.500156,0.430822,0.413025
min,1.000000,90000.000000,90000.000000,180000.000000,3.000000,68.000000,5.000000,6.000000,2025.000000,1.000000,0.000000,0.000000,0.000000
25%,4.000000,144750.000000,120000.000000,218000.000000,8.200000,82.000000,15.375000,6.900000,2025.000000,2.000000,0.000000,1.000000,0.000000
50%,6.000000,180000.000000,140000.000000,270000.000000,10.500000,89.000000,23.400000,7.900000,2026.000000,3.000000,0.000000,1.000000,0.000000
75%,8.000000,236250.000000,155000.000000,290000.000000,14.200000,95.000000,42.700000,8.900000,2026.000000,5.000000,1.000000,1.000000,0.000000
max,15.000000,384000.000000,180000.000000,320000.000000,18.000000,98.000000,87.800000,9.800000,2026.000000,12.000000,1.000000,1.000000,1.000000


# Feature Engineering

In [49]:
for col in df.columns:
    if df[col].dtype == 'str':
        print(f"Unique values in '{col}': {df[col].unique()}\n\n")
        df[col] = df[col].apply(lambda x: x.lower())

Unique values in 'job_id': <StringArray>
['AIJOB0001', 'AIJOB0002', 'AIJOB0003', 'AIJOB0004', 'AIJOB0005', 'AIJOB0006',
 'AIJOB0007', 'AIJOB0008', 'AIJOB0009', 'AIJOB0010',
 ...
 'AIJOB1491', 'AIJOB1492', 'AIJOB1493', 'AIJOB1494', 'AIJOB1495', 'AIJOB1496',
 'AIJOB1497', 'AIJOB1498', 'AIJOB1499', 'AIJOB1500']
Length: 1500, dtype: str


Unique values in 'job_title': <StringArray>
[      'AI Agent Developer',          'Prompt Engineer',
             'LLM Engineer',       'Data Engineer (AI)',
       'AI Product Manager',     'AI Security Engineer',
       'Senior ML Engineer',             'NLP Engineer',
   'AI Solutions Architect',              'ML Engineer',
   'Generative AI Engineer',   'Deep Learning Engineer',
   'Multimodal AI Engineer',           'MLOps Engineer',
      'AI Business Analyst',             'RAG Engineer',
   'Robotics Engineer (AI)',    'Senior Data Scientist',
        'AI Ethics Officer',    'AI Infrastructure Eng',
              'AI Engineer',           'Data Scie

In [63]:
# Dropping this columns to avoid {target leakage}
df_reg = df.drop(columns=['job_id', 'salary_min_usd', 'salary_max_usd', 'salary_tier','city', 'required_skills'])
df_reg.head()

,job_title,job_category,experience_level,years_of_experience,education_required,annual_salary_usd,country,remote_work,company_size,industry,ai_salary_premium_pct,demand_score,demand_growth_yoy_pct,benefits_score_10,posting_year,posting_month,is_senior,is_remote_friendly,is_llm_role
0,ai agent developer,ai engineering,senior (6-9 yrs),7,master's,239000.0,usa,on-site,startup (1-50),finance,13.1,96,16.9,6.8,2026,3,1,0,1
1,prompt engineer,ai engineering,senior (6-9 yrs),2,bachelor's,166000.0,uk,hybrid,enterprise (5000+),finance,5.4,82,11.6,6.2,2026,1,1,1,1
2,llm engineer,ai engineering,senior (6-9 yrs),4,associate's,360000.0,usa,fully remote,big tech (faang+),finance,9.1,98,42.7,7.7,2026,1,1,1,1
3,data engineer (ai),data engineering,senior (6-9 yrs),3,bachelor's,161000.0,singapore,fully remote,sme (51-500),technology,12.0,88,6.7,9.5,2026,3,1,1,0
4,ai product manager,product,lead (10+ yrs),5,bootcamp/self-taught,283000.0,usa,fully remote,enterprise (5000+),automotive,9.4,85,17.3,8.9,2026,1,1,1,0


In [ ]:
def expbucket(x):
    if x <= 2:
        return 'junior'
    elif x <= 5 and x > 2:
        return 'mid'
    elif x <= 10 and x > 5:
        return 'senior'
    else:
        return 'lead'
    
def encode_remote(x):
    if 'remote' in x.lower():
        return 1
    else:
        return 0

# Use label encoder for ordinal features or for output (only in classification)
exp_order = {'junior': 0, 'mid': 1, 'senior': 2, 'lead': 3}
edu_order = {'bootcamp/self-taught': 0, 'associate': 1, 'bachelor': 2, 'master': 3, 'phd': 4}
company_size_order = {'startup': 0, 'sme': 1, 'mid': 2, 'big': 4, 'enterprise': 3}

df_reg['years_of_experience'] = df_reg['years_of_experience'].apply(expbucket)
df_reg['experience_level'] = df_reg['experience_level'].apply(lambda x: x.split(' ')[0])
df_reg['company_size'] = df_reg['company_size'].apply(lambda x: x.split(' ')[0])
df_reg['remote_work'] = df_reg['remote_work'].apply(encode_remote)
df_reg['education_required'] = df_reg['education_required'].apply(lambda x: x.split("'")[0])
df_reg['years_of_experience'] = df_reg['years_of_experience'].map(exp_order)
df_reg['experience_level'] = df_reg['experience_level'].map(exp_order)
df_reg['education_required'] = df_reg['education_required'].map(edu_order)
df_reg['company_size'] = df_reg['company_size'].map(company_size_order)

df_reg.head()

,job_title,job_category,experience_level,years_of_experience,education_required,annual_salary_usd,country,remote_work,company_size,industry,ai_salary_premium_pct,demand_score,demand_growth_yoy_pct,benefits_score_10,posting_year,posting_month,is_senior,is_remote_friendly,is_llm_role
0,ai agent developer,ai engineering,2.0,2,3,239000.0,usa,0,0.0,finance,13.1,96,16.9,6.8,2026,3,1,0,1
1,prompt engineer,ai engineering,2.0,0,2,166000.0,uk,0,3.0,finance,5.4,82,11.6,6.2,2026,1,1,1,1
2,llm engineer,ai engineering,2.0,1,1,360000.0,usa,1,4.0,finance,9.1,98,42.7,7.7,2026,1,1,1,1
3,data engineer (ai),data engineering,2.0,1,2,161000.0,singapore,1,1.0,technology,12.0,88,6.7,9.5,2026,3,1,1,0
4,ai product manager,product,3.0,1,0,283000.0,usa,1,3.0,automotive,9.4,85,17.3,8.9,2026,1,1,1,0


In [70]:
encode_categories = OneHotEncoder()
for col in df_reg.columns:
    if df_reg[col].dtype == 'str':
        encoded = encode_categories.fit_transform(df_reg[[col]])
        df_reg = pd.concat([df_reg, pd.DataFrame(encoded.toarray(), columns=encode_categories.get_feature_names_out([col]))], axis=1)
        df_reg.drop(columns=[col], inplace=True)
df_reg.head()

,experience_level,years_of_experience,education_required,annual_salary_usd,remote_work,company_size,ai_salary_premium_pct,demand_score,demand_growth_yoy_pct,benefits_score_10,posting_year,posting_month,is_senior,is_remote_friendly,is_llm_role,job_title_ai agent developer,job_title_ai business analyst,job_title_ai compliance manager,job_title_ai engineer,job_title_ai ethics officer,job_title_ai infrastructure eng,job_title_ai product manager,job_title_ai research scientist,job_title_ai security engineer,job_title_ai solutions architect,job_title_computer vision engineer,job_title_data engineer (ai),job_title_data scientist,job_title_deep learning engineer,job_title_generative ai engineer,job_title_llm engineer,job_title_ml engineer,job_title_mlops engineer,job_title_multimodal ai engineer,job_title_nlp engineer,job_title_prompt engineer,job_title_rag engineer,job_title_robotics engineer (ai),job_title_senior data scientist,job_title_senior ml engineer,job_category_ai engineering,job_category_architecture,job_category_business,job_category_data engineering,job_category_data science,job_category_governance,job_category_infrastructure,job_category_ml operations,job_category_product,job_category_research,job_category_robotics,job_category_security,country_australia,country_canada,country_china,country_france,country_germany,country_global,country_india,country_japan,country_netherlands,country_singapore,country_switzerland,country_uae,country_uk,country_usa,industry_automotive,industry_consulting,industry_education,industry_energy,industry_finance,industry_government,industry_healthcare,industry_manufacturing,industry_media,industry_research,industry_retail,industry_technology
0,2.0,2,3,239000.0,0,0.0,13.1,96,16.9,6.8,2026,3,1,0,1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2.0,0,2,166000.0,0,3.0,5.4,82,11.6,6.2,2026,1,1,1,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2.0,1,1,360000.0,1,4.0,9.1,98,42.7,7.7,2026,1,1,1,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2.0,1,2,161000.0,1,1.0,12.0,88,6.7,9.5,2026,3,1,1,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,3.0,1,0,283000.0,1,3.0,9.4,85,17.3,8.9,2026,1,1,1,0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
